# Week 08 · Wednesday — CNNs + Embeddings
**PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar**

**Scenario:** Meera Nair (Head of Trust & Safety) needs:
1. A content classifier to identify harmful posts.
2. A semantic search system to surface conceptually similar posts.

**Datasets:** `social_media_posts.csv` | MNIST (via torchvision)

## Environment Setup

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings
import json

# ── Numerical / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_curve, average_precision_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ── Sentence-Transformers ─────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
print('Environment ready.')

---
## Sub-step 1 (Easy — Required)
### Load and Characterise `social_media_posts.csv`

In [ ]:
SOCIAL_CSV_PATH = 'social_media_posts.csv'  # Place in same folder or update path


def load_social_data(path: str) -> pd.DataFrame:
    """Load social media posts CSV with basic validation."""
    try:
        df = pd.read_csv(path)
        print(f'Loaded {len(df):,} rows, {df.shape[1]} columns.')
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f'Dataset not found at: {path}. Download from LMS.')


def characterise_social_data(df: pd.DataFrame) -> dict:
    """Full EDA of the social media dataset. Returns a summary dict."""
    summary = {}

    # --- Schema ---
    print('\n=== Schema ===')
    print(df.dtypes)
    summary['columns'] = df.columns.tolist()

    # --- Missingness ---
    print('\n=== Missing values ===')
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.any() else 'No missing values.')
    summary['missing'] = missing.to_dict()

    # --- Class distribution: hate_speech ---
    print('\n=== hate_speech distribution ===')
    hs_counts = df['hate_speech'].value_counts()
    hs_ratio  = df['hate_speech'].value_counts(normalize=True)
    print(pd.concat([hs_counts, hs_ratio.rename('proportion')], axis=1))
    summary['hate_speech_dist'] = hs_counts.to_dict()

    # --- Class distribution: spam ---
    print('\n=== spam distribution ===')
    spam_counts = df['spam'].value_counts()
    spam_ratio  = df['spam'].value_counts(normalize=True)
    print(pd.concat([spam_counts, spam_ratio.rename('proportion')], axis=1))
    summary['spam_dist'] = spam_counts.to_dict()

    # --- Language breakdown ---
    print('\n=== Language distribution ===')
    print(df['language'].value_counts())

    # --- Platform breakdown ---
    print('\n=== Platform distribution ===')
    print(df['platform'].value_counts())

    # --- Engagement columns ---
    engagement_cols = [c for c in df.columns if c in ['likes', 'shares', 'comments', 'retweets']]
    if engagement_cols:
        print('\n=== Engagement stats ===')
        print(df[engagement_cols].describe())

    return summary


def plot_class_distributions(df: pd.DataFrame) -> None:
    """Visualise class imbalance for hate_speech and spam flags."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, col in zip(axes, ['hate_speech', 'spam']):
        counts = df[col].value_counts()
        ax.bar(counts.index.astype(str), counts.values, color=['steelblue', 'tomato'])
        ax.set_title(f'{col} — class distribution')
        ax.set_xlabel('Class label')
        ax.set_ylabel('Count')
        for i, v in enumerate(counts.values):
            ax.text(i, v + 10, f'{v:,}', ha='center', fontsize=10)

    plt.tight_layout()
    plt.savefig('class_distributions.png', dpi=120)
    plt.show()


# ── Run ───────────────────────────────────────────────────────────────────────
df_social = load_social_data(SOCIAL_CSV_PATH)
social_summary = characterise_social_data(df_social)
plot_class_distributions(df_social)

In [ ]:
# ── Evaluation implications of class imbalance ────────────────────────────────

def document_imbalance_implications(summary: dict) -> None:
    """
    Document why class imbalance affects evaluation choices.
    Explains what metrics will be used in Sub-step 5 and why.
    """
    hs_dist = summary['hate_speech_dist']
    total   = sum(hs_dist.values())
    minority_frac = min(hs_dist.values()) / total

    print('=== Class Imbalance Analysis & Evaluation Implications ===')
    print(f'Minority class (hate=1) fraction: {minority_frac:.1%}')
    print()
    print('Problem created for classifiers:')
    print('  - A naive majority-class classifier achieves high accuracy'
          ' by predicting 0 for everything.')
    print('  - Accuracy is therefore a misleading metric here.')
    print()
    print('Evaluation choices for Sub-step 5:')
    print('  - PRIMARY:   Macro-F1 / F1 on minority class')
    print('               Balances precision & recall without favouring majority.')
    print('  - SECONDARY: Average Precision (area under PR curve)')
    print('               Directly measures retrieval quality for the minority class.')
    print('  - AVOID:     Accuracy — inflated by class majority.')
    print('  - CLASS WEIGHT strategy will be applied in Sub-step 4 to handle imbalance.')


document_imbalance_implications(social_summary)

---
## Sub-step 2 (Easy — Required)
### Load and Characterise MNIST

In [ ]:
def load_mnist_datasets(data_root: str = './data') -> tuple:
    """Download/load MNIST via torchvision and return train/test datasets."""
    transform = transforms.Compose([
        transforms.ToTensor(),          # scales [0,255] -> [0.0,1.0]
    ])
    try:
        train_ds = datasets.MNIST(root=data_root, train=True,  download=True, transform=transform)
        test_ds  = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
        print(f'Train samples: {len(train_ds):,} | Test samples: {len(test_ds):,}')
        return train_ds, test_ds
    except Exception as e:
        raise RuntimeError(f'Failed to load MNIST: {e}')


def characterise_mnist(train_ds, test_ds) -> dict:
    """Describe MNIST structure and report any preprocessing requirements."""
    sample_img, sample_label = train_ds[0]

    print(f'Image shape (C x H x W): {tuple(sample_img.shape)}')
    print(f'Pixel value range:        [{sample_img.min():.2f}, {sample_img.max():.2f}]')
    print(f'Dtype:                     {sample_img.dtype}')

    # Class distribution
    train_labels = [train_ds[i][1] for i in range(len(train_ds))]
    label_counts = pd.Series(train_labels).value_counts().sort_index()
    print('\n=== Digit class distribution (train) ===')
    print(label_counts.to_string())

    print('\n=== Preprocessing required? ===')
    print('  - Pixel scaling [0,255]->[0,1]:  DONE via ToTensor().')
    print('  - Normalisation with MNIST mean/std: APPLIED below for training stability.')
    print('  - Class balance: MNIST is well-balanced (~6,000 samples/digit). No resampling needed.')
    print('  - Image size (28x28): consistent — no padding or resizing needed.')

    return {'shape': tuple(sample_img.shape), 'label_counts': label_counts.to_dict()}


def plot_mnist_samples(train_ds, n_per_class: int = 2) -> None:
    """Show sample images and class distribution for MNIST."""
    fig, axes = plt.subplots(10, n_per_class, figsize=(n_per_class * 1.5, 15))
    class_indices = {k: [] for k in range(10)}
    for idx, (_, lbl) in enumerate(train_ds):
        if len(class_indices[lbl]) < n_per_class:
            class_indices[lbl].append(idx)
        if all(len(v) == n_per_class for v in class_indices.values()):
            break
    for digit in range(10):
        for col, idx in enumerate(class_indices[digit]):
            img, _ = train_ds[idx]
            axes[digit][col].imshow(img.squeeze(), cmap='gray')
            axes[digit][col].axis('off')
            if col == 0:
                axes[digit][col].set_ylabel(str(digit), fontsize=12)
    plt.suptitle('MNIST samples by digit class', fontsize=14)
    plt.tight_layout()
    plt.savefig('mnist_samples.png', dpi=120)
    plt.show()


# ── Run ───────────────────────────────────────────────────────────────────────
MNIST_MEAN, MNIST_STD = (0.1307,), (0.3081,)  # well-known MNIST statistics

train_ds_raw, test_ds_raw = load_mnist_datasets()
mnist_info = characterise_mnist(train_ds_raw, test_ds_raw)
plot_mnist_samples(train_ds_raw)

In [ ]:
def build_normalised_mnist_loaders(
    data_root: str = './data',
    batch_size: int = 64
) -> tuple:
    """Build DataLoaders with mean/std normalisation applied."""
    norm_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MNIST_MEAN, MNIST_STD),
    ])
    train_ds = datasets.MNIST(root=data_root, train=True,  download=False, transform=norm_transform)
    test_ds  = datasets.MNIST(root=data_root, train=False, download=False, transform=norm_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0)

    print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')
    return train_loader, test_loader


train_loader, test_loader = build_normalised_mnist_loaders()
print('MNIST ready for Sub-step 3.')

---
## Sub-step 3 (Medium — Required)
### Build, Train, and Analyse a CNN on MNIST

In [ ]:
class MNISTConvNet(nn.Module):
    """
    Two-block CNN for MNIST digit classification.

    Architecture rationale:
      Block 1: 1->16 filters (3x3) — learns low-level edges and curves.
      Block 2: 16->32 filters (3x3) — learns mid-level strokes and digit parts.
      Each block: Conv -> ReLU -> MaxPool(2x2) -> BatchNorm.
      Head: Flatten -> FC(128) -> Dropout(0.4) -> FC(10).
      Dropout prevents overfitting on the small MNIST feature space.
    """

    def __init__(self, num_classes: int = 10, dropout_rate: float = 0.4):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.BatchNorm2d(16),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.BatchNorm2d(32),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')
model = MNISTConvNet().to(DEVICE)
print(model)

In [ ]:
NUM_EPOCHS      = 5
LEARNING_RATE   = 1e-3
WEIGHT_DECAY    = 1e-4


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimiser: optim.Optimizer,
    device: torch.device
) -> float:
    """Run one training epoch; return mean loss."""
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimiser.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()
        running_loss += loss.item()
    return running_loss / len(loader)


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device
) -> float:
    """Compute accuracy on a DataLoader."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total


criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimiser, step_size=3, gamma=0.5)

history = {'train_loss': [], 'test_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    loss = train_one_epoch(model, train_loader, criterion, optimiser, DEVICE)
    acc  = evaluate_model(model, test_loader, DEVICE)
    scheduler.step()
    history['train_loss'].append(loss)
    history['test_acc'].append(acc)
    print(f'Epoch {epoch}/{NUM_EPOCHS}  loss={loss:.4f}  test_acc={acc:.4f}')

# ── Training curve ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], marker='o')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[1].plot(history['test_acc'], marker='o', color='green')
axes[1].set_title('Test Accuracy')
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('cnn_training_curve.png', dpi=120)
plt.show()

In [ ]:
def visualise_conv1_filters(model: nn.Module, n_filters: int = 16) -> None:
    """
    Visualise learned filters from the first Conv2d layer.
    Each 3x3 filter represents what the network looks for at the pixel level.
    """
    # Extract conv weights from block1 (first module)
    conv1_weights = model.block1[0].weight.detach().cpu().numpy()  # shape: (16, 1, 3, 3)

    fig, axes = plt.subplots(2, 8, figsize=(14, 4))
    for i, ax in enumerate(axes.flat):
        if i < n_filters:
            filt = conv1_weights[i, 0]           # single-channel filter (3x3)
            vmin, vmax = filt.min(), filt.max()
            ax.imshow(filt, cmap='RdBu_r', vmin=vmin, vmax=vmax)
            ax.set_title(f'F{i}', fontsize=8)
        ax.axis('off')

    plt.suptitle('Learned Filters — Conv Layer 1', fontsize=13)
    plt.tight_layout()
    plt.savefig('conv1_filters.png', dpi=120)
    plt.show()

    print()
    print('=== What has the network learned to detect? ===')
    print(
        'The 3x3 filters in the first convolution layer represent pixel-level feature detectors.\n'
        'Common patterns visible after training on MNIST:\n'
        '  - Horizontal edge detectors (bright top row, dark bottom row, or vice versa).\n'
        '  - Vertical edge detectors (bright left column, dark right column).\n'
        '  - Diagonal gradient detectors (used to identify curves and slants in strokes).\n'
        '  - Blob/centre detectors (bright centre, dark surround — identifies ink density).\n'
        'These low-level features correspond to the basic strokes that compose digits.\n'
        'Layer 2 composes these edges into higher-order stroke segments and loop shapes.\n'
        'This hierarchy — pixel -> edge -> stroke -> digit — mirrors human perceptual hierarchy.\n'
        'Critically (for Sub-step 6): these are LEARNED dense representations, not hand-crafted.\n'
        'TF-IDF lacks this compositional hierarchy — it operates on token co-occurrence, not structure.'
    )


visualise_conv1_filters(model)

---
## Sub-step 4 (Medium — Required)
### Hate Speech Detector + Semantic Similarity System

In [ ]:
# ── Text cleaning ─────────────────────────────────────────────────────────────
import re

def clean_text(text: str) -> str:
    """Lowercase, strip URLs, mentions, special chars; collapse whitespace."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'@\w+', '', text)              # remove mentions
    text = re.sub(r'[^a-z\s]', ' ', text)         # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()      # collapse whitespace
    return text


def prepare_text_features(df: pd.DataFrame, text_col: str = 'post_text') -> pd.DataFrame:
    """Apply text cleaning and validate the column exists."""
    if text_col not in df.columns:
        # Fallback: try common alternatives
        for alt in ['text', 'content', 'tweet', 'post']:
            if alt in df.columns:
                text_col = alt
                break
        else:
            raise KeyError(f'Text column not found. Available: {df.columns.tolist()}')

    df = df.copy()
    df['clean_text'] = df[text_col].apply(clean_text)
    print(f'Text column used: "{text_col}"')
    print(f'Empty texts after cleaning: {(df["clean_text"] == "").sum()}')
    return df


df_social = prepare_text_features(df_social)

In [ ]:
# ── Hate speech classifier with class-weight balancing ─────────────────────────

def build_hate_speech_classifier(df: pd.DataFrame) -> tuple:
    """
    TF-IDF + Logistic Regression hate speech classifier.
    Addresses class imbalance via class_weight='balanced'.
    Returns: (vectoriser, classifier, X_test_tfidf, y_test, test_indices)
    """
    # Filter out empty texts
    valid_mask = df['clean_text'].str.len() > 0
    df_valid   = df[valid_mask].reset_index(drop=True)

    X = df_valid['clean_text'].values
    y = df_valid['hate_speech'].values

    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X, y, np.arange(len(X)),
        test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    # TF-IDF features
    vectoriser = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2), min_df=2)
    X_train_tfidf = vectoriser.fit_transform(X_train)
    X_test_tfidf  = vectoriser.transform(X_test)

    # Logistic Regression with class weights
    clf = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE,
        C=1.0
    )
    clf.fit(X_train_tfidf, y_train)

    return vectoriser, clf, X_test_tfidf, y_test, idx_test, df_valid


vectoriser_hs, clf_hs, X_test_hs, y_test_hs, idx_test_hs, df_valid = \
    build_hate_speech_classifier(df_social)

y_pred_hs   = clf_hs.predict(X_test_hs)
y_scores_hs = clf_hs.predict_proba(X_test_hs)[:, 1]

print('\n=== Hate Speech Classifier — Test Set Report ===')
print(classification_report(y_test_hs, y_pred_hs, target_names=['Not Hate', 'Hate']))

In [ ]:
# ── Semantic similarity system using Sentence-BERT ────────────────────────────

SBERT_MODEL_NAME = 'all-MiniLM-L6-v2'   # fast, good quality


def build_sbert_index(texts: list, model_name: str = SBERT_MODEL_NAME) -> tuple:
    """Encode all texts into SBERT embeddings. Returns (model, embedding matrix)."""
    print(f'Loading SBERT model: {model_name}')
    sbert = SentenceTransformer(model_name)
    print(f'Encoding {len(texts):,} texts...')
    embeddings = sbert.encode(texts, batch_size=128, show_progress_bar=True,
                               convert_to_numpy=True, normalize_embeddings=True)
    print(f'Embedding matrix shape: {embeddings.shape}')
    return sbert, embeddings


def semantic_search(
    query_text: str,
    sbert_model,
    embedding_matrix: np.ndarray,
    df_corpus: pd.DataFrame,
    text_col: str = 'clean_text',
    top_k: int = 10
) -> pd.DataFrame:
    """
    Retrieve the top-k most semantically similar posts to a query.
    Returns a DataFrame with similarity scores.
    """
    query_vec = sbert_model.encode(
        [query_text], normalize_embeddings=True, convert_to_numpy=True
    )
    # Cosine similarity: since embeddings are L2-normalised, dot product == cosine sim
    scores = (embedding_matrix @ query_vec.T).squeeze()
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = df_corpus.iloc[top_indices].copy()
    results['similarity_score'] = scores[top_indices]
    return results


# Build index on the full valid corpus
all_texts = df_valid['clean_text'].tolist()
sbert_model, sbert_embeddings = build_sbert_index(all_texts)

In [ ]:
# ── Demo: find semantically similar posts to a known hate speech example ───────

def demo_semantic_search(df_corpus: pd.DataFrame, sbert_model, embeddings: np.ndarray) -> pd.DataFrame:
    """Pick first hate speech post as query, retrieve top 10 similar posts."""
    hate_posts = df_corpus[df_corpus['hate_speech'] == 1]
    if len(hate_posts) == 0:
        print('No hate speech posts found in corpus.')
        return pd.DataFrame()

    query_row  = hate_posts.iloc[0]
    query_text = query_row['clean_text']
    print(f'Query post:\n  "{query_text[:200]}"\n')

    results = semantic_search(
        query_text=query_text,
        sbert_model=sbert_model,
        embedding_matrix=embeddings,
        df_corpus=df_corpus,
        top_k=10
    )

    print('=== Top-10 Semantically Similar Posts ===')
    display_cols = ['clean_text', 'hate_speech', 'similarity_score']
    existing_display_cols = [c for c in display_cols if c in results.columns]
    print(results[existing_display_cols].to_string(index=False))
    return results


similar_posts = demo_semantic_search(df_valid, sbert_model, sbert_embeddings)

---
## Sub-step 5 (Medium — Required)
### Two-Stage Moderation Pipeline + Evaluation

In [ ]:
SIMILARITY_THRESHOLD = 0.75    # tunable threshold for Stage 2 retrieval
DAILY_POST_VOLUME    = 100_000


def run_stage1_classifier(
    clf,
    vectoriser,
    texts: list,
    threshold: float = 0.5
) -> np.ndarray:
    """
    Stage 1: TF-IDF + LR hate speech classifier.
    Returns binary predictions (0 = safe, 1 = flagged).
    """
    features = vectoriser.transform(texts)
    probs    = clf.predict_proba(features)[:, 1]
    return (probs >= threshold).astype(int)


def run_stage2_semantic_retrieval(
    stage1_flags: np.ndarray,
    sbert_model,
    embeddings: np.ndarray,
    df_corpus: pd.DataFrame,
    similarity_threshold: float = SIMILARITY_THRESHOLD
) -> np.ndarray:
    """
    Stage 2: For each post flagged by Stage 1, retrieve semantically
    similar posts and flag them too.
    Returns an updated flags array with Stage 2 additions.
    """
    stage2_flags = stage1_flags.copy()
    flagged_indices = np.where(stage1_flags == 1)[0]

    for idx in flagged_indices:
        query_text = df_corpus.iloc[idx]['clean_text']
        query_vec  = sbert_model.encode(
            [query_text], normalize_embeddings=True, convert_to_numpy=True
        )
        scores = (embeddings @ query_vec.T).squeeze()
        similar_mask = (scores >= similarity_threshold)
        stage2_flags[similar_mask] = 1

    return stage2_flags


def evaluate_pipeline(
    y_true: np.ndarray,
    stage1_preds: np.ndarray,
    stage2_preds: np.ndarray
) -> dict:
    """
    Compare Stage 1 and combined Stage 1+2 pipeline.
    Primary metric: F1 on hate class (label=1).
    Secondary: Average Precision (area under PR curve).
    """
    metrics = {}

    for name, preds in [('Stage 1 only', stage1_preds), ('Stage 1+2', stage2_preds)]:
        f1  = f1_score(y_true, preds, pos_label=1, zero_division=0)
        rep = classification_report(y_true, preds, target_names=['Not Hate', 'Hate'])
        metrics[name] = {'f1_hate': f1, 'report': rep}
        print(f'\n=== {name} ===')
        print(rep)

    additional = int((stage2_preds - stage1_preds).clip(min=0).sum())
    true_additional = int(((stage2_preds == 1) & (stage1_preds == 0) & (y_true == 1)).sum())

    print(f'Additional posts flagged by Stage 2:           {additional:,}')
    print(f'Of those, confirmed hate (y_true=1):           {true_additional:,}')

    metrics['additional_flagged']     = additional
    metrics['additional_true_hates']  = true_additional
    return metrics


# ── Run the pipeline on the test split ────────────────────────────────────────
test_texts  = df_valid.iloc[idx_test_hs]['clean_text'].tolist()
test_embeds = sbert_embeddings[idx_test_hs]
test_df     = df_valid.iloc[idx_test_hs].reset_index(drop=True)

stage1_preds = run_stage1_classifier(clf_hs, vectoriser_hs, test_texts)

stage2_preds = run_stage2_semantic_retrieval(
    stage1_flags=stage1_preds,
    sbert_model=sbert_model,
    embeddings=test_embeds,
    df_corpus=test_df
)

pipeline_metrics = evaluate_pipeline(y_test_hs, stage1_preds, stage2_preds)

In [ ]:
def present_recommendation(metrics: dict, daily_volume: int = DAILY_POST_VOLUME) -> None:
    """
    Present pipeline results as an actionable recommendation for Meera's team.
    """
    test_size    = len(y_test_hs)
    stage2_rate  = metrics['additional_flagged'] / max(test_size, 1)
    extra_daily  = int(stage2_rate * daily_volume)
    stage1_flags = int(stage1_preds.sum())
    stage1_rate  = stage1_flags / max(test_size, 1)
    stage1_daily = int(stage1_rate * daily_volume)

    print('===================================================================')
    print('  RECOMMENDATION — Two-Stage Moderation Pipeline')
    print('===================================================================')
    print(f'  Metric used: F1 on hate class (justification below).')
    print()
    print(f'  Stage 1 F1 (hate class): {metrics["Stage 1 only"]["f1_hate"]:.3f}')
    print(f'  Stage 1+2 F1 (hate cls): {metrics["Stage 1+2"]["f1_hate"]:.3f}')
    print()
    print(f'  Estimated daily review volume at {daily_volume:,} posts/day:')
    print(f'    Stage 1 alone:         ~{stage1_daily:,} posts/day for human review')
    print(f'    Stage 2 addition:      ~{extra_daily:,} additional posts/day')
    print(f'    Total (Stage 1+2):     ~{stage1_daily + extra_daily:,} posts/day')
    print()
    print('  Metric justification:')
    print('    F1 on the hate class is the primary metric because:')
    print('      1. The dataset is imbalanced — accuracy is misleading.')
    print('      2. Both false negatives (missed hate) and false positives')
    print('         (reviewer burden) carry real cost for Trust & Safety.')
    print('      3. F1 balances precision and recall, reflecting both harms.')
    print('      4. Average Precision is secondary for ranking/threshold tuning.')
    print('===================================================================')


present_recommendation(pipeline_metrics)

---
## Sub-step 6 (Hard — Optional, required for Band 4)
### TF-IDF vs Sentence Embeddings: Empirical Comparison

In [ ]:
def build_tfidf_index(texts: list, max_features: int = 20_000) -> tuple:
    """Fit TF-IDF vectoriser and compute normalised document matrix."""
    tfidf = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), min_df=2)
    matrix = tfidf.fit_transform(texts)  # sparse (n_docs, vocab)
    return tfidf, matrix


def tfidf_search(
    query_text: str,
    tfidf_vectoriser,
    tfidf_matrix,
    df_corpus: pd.DataFrame,
    top_k: int = 10
) -> pd.DataFrame:
    """Retrieve top-k similar posts using TF-IDF cosine similarity."""
    query_vec = tfidf_vectoriser.transform([query_text])
    scores    = cosine_similarity(query_vec, tfidf_matrix).squeeze()
    top_idx   = np.argsort(scores)[::-1][:top_k]
    results   = df_corpus.iloc[top_idx].copy()
    results['tfidf_score'] = scores[top_idx]
    return results


def sbert_search_topk(
    query_text: str,
    sbert_model,
    sbert_embeddings: np.ndarray,
    df_corpus: pd.DataFrame,
    top_k: int = 10
) -> pd.DataFrame:
    """Retrieve top-k similar posts using SBERT cosine similarity."""
    q_vec  = sbert_model.encode([query_text], normalize_embeddings=True, convert_to_numpy=True)
    scores = (sbert_embeddings @ q_vec.T).squeeze()
    top_idx = np.argsort(scores)[::-1][:top_k]
    results = df_corpus.iloc[top_idx].copy()
    results['sbert_score'] = scores[top_idx]
    return results


def compare_retrieval_systems(
    queries: list,
    tfidf_vectoriser,
    tfidf_matrix,
    sbert_model,
    sbert_embeddings: np.ndarray,
    df_corpus: pd.DataFrame,
    top_k: int = 10
) -> None:
    """For each query, compare TF-IDF vs SBERT top-k sets and overlap."""
    for i, query in enumerate(queries):
        print(f'\n--- Query {i+1}: "{query[:80]}..." ---')

        tfidf_res = tfidf_search(query, tfidf_vectoriser, tfidf_matrix, df_corpus, top_k)
        sbert_res = sbert_search_topk(query, sbert_model, sbert_embeddings, df_corpus, top_k)

        tfidf_set = set(tfidf_res.index)
        sbert_set = set(sbert_res.index)

        overlap = tfidf_set & sbert_set
        jaccard = len(overlap) / len(tfidf_set | sbert_set)

        print(f'  TF-IDF top-{top_k} indices: {sorted(tfidf_set)}')
        print(f'  SBERT  top-{top_k} indices: {sorted(sbert_set)}')
        print(f'  Overlap:  {len(overlap)} posts | Jaccard: {jaccard:.3f}')

        sbert_only = sbert_set - tfidf_set
        if sbert_only:
            print(f'  Posts found by SBERT but NOT TF-IDF: {sorted(sbert_only)}')
            sbert_unique_texts = df_corpus.loc[list(sbert_only), 'clean_text'].tolist()
            for t in sbert_unique_texts[:3]:
                print(f'    -> "{t[:100]}"')


# ── Build TF-IDF index ────────────────────────────────────────────────────────
tfidf_vec, tfidf_mat = build_tfidf_index(df_valid['clean_text'].tolist())

# Select 3 known hate speech posts as queries
hate_sample_queries = df_valid[df_valid['hate_speech'] == 1]['clean_text'].head(3).tolist()

compare_retrieval_systems(
    queries=hate_sample_queries,
    tfidf_vectoriser=tfidf_vec,
    tfidf_matrix=tfidf_mat,
    sbert_model=sbert_model,
    sbert_embeddings=sbert_embeddings,
    df_corpus=df_valid
)

In [ ]:
def analyse_tfidf_vs_sbert_difference() -> None:
    """
    Explain WHY embedding-based similarity captures something TF-IDF cannot,
    linking back to CNN filter observations from Sub-step 3.
    """
    print('=== Why SBERT Captures What TF-IDF Cannot ===')
    print()
    print('TF-IDF limitations:')
    print('  - TF-IDF is a bag-of-words model. Similarity is driven by shared tokens.')
    print('  - Two posts saying "I hate X" and "I despise X" share no tokens,')
    print('    so TF-IDF assigns them near-zero similarity.')
    print('  - It cannot detect paraphrasing, synonym use, or deliberate wording rotation.')
    print()
    print('SBERT advantages:')
    print('  - SBERT produces dense contextual embeddings from a transformer trained')
    print('    on millions of sentence pairs to cluster semantically similar content.')
    print('  - Synonyms, paraphrases, and topic-shifted wording map to close points')
    print('    in embedding space — critical for detecting coordinated campaigns.')
    print()
    print('Connection to Sub-step 3 — CNN filter visualisations:')
    print('  - The CNN learned HIERARCHICAL representations: pixel -> edge -> stroke -> digit.')
    print('  - A CNN does not match pixels; it matches structural features.')
    print('  - Similarly, SBERT does not match tokens; it matches semantic structures.')
    print('  - Both demonstrate that LEARNED dense representations capture compositional')
    print('    structure that hand-crafted (TF-IDF / pixel matching) features miss.')
    print('  - TF-IDF is the text equivalent of pixel-level image matching —')
    print('    exact token overlap, no abstraction.')
    print('    SBERT is the text equivalent of the CNN feature space —')
    print('    learned, distributed, semantics-aware.')


analyse_tfidf_vs_sbert_difference()

---
## Sub-step 7 (Hard — Optional, required for Band 4)
### Transfer Learning Experiment: MNIST CNN Features on Social Media Data

In [ ]:
from PIL import Image, ImageDraw, ImageFont


def text_to_image_thumbnail(text: str, img_size: int = 28) -> np.ndarray:
    """
    Render a short text string as a grayscale pixel image (28x28).
    This creates a 'spectrogram-like' visual representation of text metadata.
    The CNN trained on MNIST will process these as if they were digit images.
    """
    img = Image.new('L', (img_size, img_size), color=0)  # black background
    draw = ImageDraw.Draw(img)
    # Truncate text to fit; render at tiny scale
    snippet = (text or '')[:20].ljust(20)
    draw.text((1, 8), snippet[:10], fill=255)
    draw.text((1, 16), snippet[10:20], fill=200)
    return np.array(img, dtype=np.float32) / 255.0


def extract_cnn_features(
    model: nn.Module,
    texts: list,
    device: torch.device,
    batch_size: int = 64
) -> np.ndarray:
    """
    Render texts as 28x28 grayscale thumbnails,
    pass through the MNIST CNN up to the penultimate layer,
    and return the 128-dim feature vectors.
    """
    # Build a feature extractor that stops before the final classifier head
    feature_extractor = nn.Sequential(
        model.block1,
        model.block2,
        nn.Flatten(),
        model.classifier[1],  # Linear(32*7*7, 128)
        model.classifier[2],  # ReLU
    ).to(device)
    feature_extractor.eval()

    all_feats = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i: i + batch_size]
        imgs = np.stack([text_to_image_thumbnail(t) for t in batch_texts])
        # Shape: (B, 28, 28) -> (B, 1, 28, 28)
        tensor = torch.tensor(imgs).unsqueeze(1).to(device)
        with torch.no_grad():
            feats = feature_extractor(tensor).cpu().numpy()
        all_feats.append(feats)

    return np.concatenate(all_feats, axis=0)


print('Extracting CNN features from social media text thumbnails...')
social_texts  = df_valid['clean_text'].tolist()
social_labels = df_valid['hate_speech'].values

cnn_features = extract_cnn_features(model, social_texts, DEVICE)
print(f'Feature matrix shape: {cnn_features.shape}')

In [ ]:
def run_transfer_experiment(
    features: np.ndarray,
    labels: np.ndarray
) -> dict:
    """
    Train a logistic regression on CNN features extracted from text thumbnails.
    Compare to a baseline trained on SBERT embeddings.
    Returns performance metrics for both.
    """
    X_tr, X_te, y_tr, y_te = train_test_split(
        features, labels, test_size=0.2,
        random_state=RANDOM_STATE, stratify=labels
    )

    clf_transfer = LogisticRegression(
        class_weight='balanced', max_iter=500, random_state=RANDOM_STATE
    )
    clf_transfer.fit(X_tr, y_tr)
    y_pred = clf_transfer.predict(X_te)
    f1_cnn = f1_score(y_te, y_pred, pos_label=1, zero_division=0)

    print('=== Transfer Learning Result: MNIST CNN Features on Hate Speech ===')
    print(classification_report(y_te, y_pred, target_names=['Not Hate', 'Hate']))
    print(f'F1 (hate class): {f1_cnn:.3f}')

    # SBERT baseline for fair comparison
    sbert_X_tr, sbert_X_te, sbert_y_tr, sbert_y_te = train_test_split(
        sbert_embeddings, labels, test_size=0.2,
        random_state=RANDOM_STATE, stratify=labels
    )
    clf_sbert = LogisticRegression(
        class_weight='balanced', max_iter=500, random_state=RANDOM_STATE
    )
    clf_sbert.fit(sbert_X_tr, sbert_y_tr)
    y_pred_sbert = clf_sbert.predict(sbert_X_te)
    f1_sbert = f1_score(sbert_y_te, y_pred_sbert, pos_label=1, zero_division=0)

    print(f'\nSBERT baseline F1 (hate class): {f1_sbert:.3f}')
    print(f'CNN transfer F1 (hate class):   {f1_cnn:.3f}')
    print(f'Difference: {f1_cnn - f1_sbert:+.3f}')

    return {'f1_cnn_transfer': f1_cnn, 'f1_sbert_baseline': f1_sbert}


transfer_results = run_transfer_experiment(cnn_features, social_labels)

In [ ]:
def analyse_transfer_results(results: dict) -> None:
    """
    Explain WHY CNN transfer from MNIST to text does or does not work.
    Addresses representation learning theory.
    """
    gap = results['f1_cnn_transfer'] - results['f1_sbert_baseline']
    worked = gap > 0.05

    print('=== Transfer Learning Analysis ===')
    print()
    print(f'Result: CNN transfer F1={results["f1_cnn_transfer"]:.3f} vs '
          f'SBERT baseline F1={results["f1_sbert_baseline"]:.3f}')
    print(f'Transfer {"HELPED" if worked else "DID NOT HELP"}.')
    print()
    print('Why CNN transfer from MNIST to text is expected to fail:')
    print()
    print('  1. DOMAIN MISMATCH: MNIST filters learned to detect ink strokes')
    print('     on a clean 28x28 canvas — centred, high-contrast, single digit.')
    print('     Text thumbnails rendered at 28x28 produce pixel patterns that')
    print('     are structurally unrelated to digit strokes.')
    print()
    print('  2. RESOLUTION vs INFORMATION DENSITY: A social media post carries')
    print('     its information in token sequences. Rendering it into 28x28')
    print('     pixels destroys most of that information.')
    print()
    print('  3. IMBALANCE COMPOUNDING: The social data is already imbalanced.')
    print('     A feature extractor that encodes noise will amplify, not reduce,')
    print('     this imbalance signal.')
    print()
    print('  4. REPRESENTATION LEARNING PRINCIPLE: Transfer succeeds when the')
    print('     source and target domains share low-level feature structure.')
    print('     Image -> Image (e.g. MNIST -> Chars74K) works.')
    print('     Image (digit strokes) -> Text (semantic content) does not work')
    print('     because the features being transferred have no semantic content')
    print('     in the target domain.')
    print()
    print('  This distinguishes engineers who understand WHAT is transferred')
    print('  (learned feature kernels for a specific input modality) from those')
    print('  who apply model.fit() without domain reasoning.')


analyse_transfer_results(transfer_results)